# Lab 1 · The control loop
MICAI 2026 · Reliable Enterprise Agents · about 25 minutes

**What you will learn**
- An agent is a loop: the model proposes a tool call, our code decides and runs it
- How the loop stops: final answer, step budget, token budget, repeated calls
- The properties any agent design must keep, whether it is a loop or not

**What we build together**
- `run_tool`: tool errors become messages for the model, never exceptions
- `run_agent`: the loop with its stop conditions

Run each cell when we reach it on the screen. Cells that start with *Before you run it* ask you to make a prediction first.

## Setup

In [ ]:
import json
import os
import random
import time
import uuid
import hashlib
from pathlib import Path

# Locally (uv) everything is already installed. In Colab we install the
# packages and download the data files from the course repository.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    import urllib.request
    subprocess.run(["pip", "install", "-q", "openai==3.41.0"], check=True)
    Path("data").mkdir(exist_ok=True)
    repo = "https://raw.githubusercontent.com/RamsesCamas/Enterprise-Agents-MICAI-2026/main/labs/data/"
    for name in ["products", "customers", "invoices", "documents"]:
        if not Path(f"data/{name}.jsonl").exists():
            urllib.request.urlretrieve(repo + f"{name}.jsonl", f"data/{name}.jsonl")

print("Data files:", sorted(p.name for p in Path("data").glob("*.jsonl")))

In [ ]:
from openai import OpenAI


def _load_dotenv():
    """Read a .env file into os.environ when running locally (not in Colab).

    Looks for a .env next to the notebook and in parent folders. Values
    already set in the real environment win, so `export GROQ_API_KEY=...`
    still takes precedence over the file.
    """
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / ".env"
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                os.environ.setdefault(key, value)
            break


def get_secret(name):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    # Locally: use the real environment first, then fall back to a .env file.
    if name not in os.environ:
        _load_dotenv()
    return os.environ.get(name)


client = OpenAI(
    base_url=os.environ.get("LLM_BASE_URL", "https://api.groq.com/openai/v1"),
    api_key=get_secret("GROQ_API_KEY") or "missing-key",
    max_retries=4,
)
MODEL_NAME = os.environ.get("LLM_MODEL", "openai/gpt-oss-120b")


def ask_model(messages, tools=None):
    """Send the conversation to the model. Return its reply as a plain dict."""
    if tools:
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, tools=tools, temperature=0)
    else:
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, temperature=0)

    message = response.choices[0].message
    reply = {"role": "assistant", "content": message.content}
    if message.tool_calls:
        reply["tool_calls"] = []
        for call in message.tool_calls:
            reply["tool_calls"].append({
                "id": call.id,
                "type": "function",
                "function": {"name": call.function.name, "arguments": call.function.arguments},
            })
    reply["tokens"] = response.usage.total_tokens
    return reply

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


class ERPError(Exception):
    def __init__(self, code, message, retryable=False, retry_after=None):
        super().__init__(message)
        self.code = code
        self.message = message
        self.retryable = retryable
        self.retry_after = retry_after


class AcmeERP:
    """A small in-memory ERP loaded from data/*.jsonl.

    It behaves like a real backend: errors with codes, idempotency keys on
    writes, and outages you can simulate with fail_next().
    """

    def __init__(self, data_dir="data"):
        self.products = {}
        for row in read_jsonl(f"{data_dir}/products.jsonl"):
            self.products[row["sku"]] = row
        self.customers = {}
        for row in read_jsonl(f"{data_dir}/customers.jsonl"):
            self.customers[row["id"]] = row
        self.invoices = {}
        for row in read_jsonl(f"{data_dir}/invoices.jsonl"):
            self.invoices[row["id"]] = row
        self.movements = []
        self.refunds = []
        self.saved_replies = {}      # idempotency key -> reply already sent
        self.pending_failures = []

    def fail_next(self, kind, times=1):
        """kind: 'unavailable', 'rate_limited', 'timeout' or 'lost_response'."""
        for _ in range(times):
            self.pending_failures.append(kind)

    def _check_failures(self):
        """Raise a simulated outage. Return True if the reply must be 'lost'."""
        if not self.pending_failures:
            return False
        kind = self.pending_failures.pop(0)
        if kind == "unavailable":
            raise ERPError("unavailable", "503 Service unavailable", retryable=True, retry_after=1.0)
        if kind == "rate_limited":
            raise ERPError("rate_limited", "429 Too many requests", retryable=True, retry_after=2.0)
        if kind == "timeout":
            raise ERPError("timeout", "No answer after 5 seconds", retryable=True)
        return kind == "lost_response"

    # ---------- reads ----------

    def get_product(self, sku):
        self._check_failures()
        if sku not in self.products:
            raise ERPError("not_found", f"Product {sku} not found")
        return dict(self.products[sku])

    def find_products(self, category=None, max_stock=None):
        self._check_failures()
        found = []
        for product in self.products.values():
            if category is not None and product["category"] != category:
                continue
            if max_stock is not None and product["stock"] > max_stock:
                continue
            found.append(dict(product))
        return found

    def get_customer(self, customer_id):
        self._check_failures()
        if customer_id not in self.customers:
            raise ERPError("not_found", f"Customer {customer_id} not found")
        return dict(self.customers[customer_id])

    def get_invoice(self, invoice_id):
        self._check_failures()
        if invoice_id not in self.invoices:
            raise ERPError("not_found", f"Invoice {invoice_id} not found")
        return dict(self.invoices[invoice_id])

    # ---------- writes: applied once per idempotency key ----------

    def adjust_stock(self, sku, delta, reason, idempotency_key=None):
        lose_reply = self._check_failures()
        if idempotency_key in self.saved_replies:
            return {**self.saved_replies[idempotency_key], "replayed": True}

        if sku not in self.products:
            raise ERPError("not_found", f"Product {sku} not found")
        product = self.products[sku]
        if product["stock"] + delta < 0:
            raise ERPError("insufficient_stock", f"Stock is {product['stock']}, cannot apply {delta}")
        product["stock"] += delta
        self.movements.append({"sku": sku, "delta": delta, "reason": reason})
        reply = {"sku": sku, "stock": product["stock"]}

        if idempotency_key:
            self.saved_replies[idempotency_key] = reply
        if lose_reply:
            # The write is committed, but the client never gets the answer.
            raise ERPError("bad_gateway", "502 Connection reset. The write may have been applied.", retryable=True)
        return reply

    def create_refund(self, invoice_id, amount, reason, idempotency_key=None):
        lose_reply = self._check_failures()
        if idempotency_key in self.saved_replies:
            return {**self.saved_replies[idempotency_key], "replayed": True}

        if invoice_id not in self.invoices:
            raise ERPError("not_found", f"Invoice {invoice_id} not found")
        invoice = self.invoices[invoice_id]
        if invoice["status"] != "paid":
            raise ERPError("invalid_state", "Only paid invoices can be refunded")
        available = invoice["total"] - invoice["refunded"]
        if amount > available:
            raise ERPError("exceeds_refundable", f"Refundable balance is {available}")
        invoice["refunded"] += amount
        self.refunds.append({"invoice_id": invoice_id, "amount": amount, "reason": reason})
        reply = {"invoice_id": invoice_id, "refunded": amount, "remaining": available - amount}

        if idempotency_key:
            self.saved_replies[idempotency_key] = reply
        if lose_reply:
            raise ERPError("bad_gateway", "502 Connection reset. The write may have been applied.", retryable=True)
        return reply


erp = AcmeERP()
print(len(erp.products), "products,", len(erp.customers), "customers,", len(erp.invoices), "invoices")

## 1. Tools
A tool has three parts:

- a **schema**: the only part the model sees
- a **function**: our code, which the model can never run by itself
- an **effect**: `read`, `write` or `irreversible`; we will use it in later labs

In [ ]:
def get_product(sku):
    return erp.get_product(sku)


def find_products(category=None, max_stock=None):
    return erp.find_products(category, max_stock)


TOOLS = {
    "get_product": {
        "function": get_product,
        "effect": "read",
        "schema": {
            "description": "Get one product: name, category, price (MXN) and stock (units).",
            "parameters": {
                "type": "object",
                "properties": {"sku": {"type": "string", "pattern": "^SKU-[0-9]{3}$"}},
                "required": ["sku"],
                "additionalProperties": False,
            },
        },
    },
    "find_products": {
        "function": find_products,
        "effect": "read",
        "schema": {
            "description": "Search products. Use max_stock to find items with low stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {"type": "string",
                                 "enum": ["computo", "perifericos", "mobiliario", "redes", "oficina", "almacenamiento"]},
                    "max_stock": {"type": "integer", "minimum": 0},
                },
                "additionalProperties": False,
            },
        },
    },
}

def tool_specs(tools):
    """What the model sees: name, description and schema. Never the function."""
    specs = []
    for name, tool in tools.items():
        specs.append({"type": "function", "function": {"name": name, **tool["schema"]}})
    return specs


print(json.dumps(tool_specs(TOOLS)[0], indent=2))

## 2. Running one tool call
The model can ask for a tool that does not exist, or send broken JSON. None of that may crash the loop:
every problem goes back to the model as a message it can read and react to.

In [ ]:
def run_tool(tool_call, tools):
    """Run one tool call and return a string for the model. Never raises."""
    name = tool_call["function"]["name"]
    if name not in tools:
        return json.dumps({"error": "unknown_tool", "message": f"There is no tool called {name}"})

    try:
        arguments = json.loads(tool_call["function"]["arguments"] or "{}")
    except json.JSONDecodeError:
        return json.dumps({"error": "invalid_json", "message": "The arguments are not valid JSON"})

    try:
        result = tools[name]["function"](**arguments)
    except ERPError as error:
        return json.dumps({"error": error.code, "message": error.message})
    except TypeError as error:
        return json.dumps({"error": "invalid_arguments", "message": str(error)})

    return json.dumps(result)

## 3. The loop
```
messages -> model -> tool calls? -- no --> final answer
              ^          | yes
              +-- result <- run_tool (our code)
```
Budgets are checked **before** each model call: once we have paid for the call, it is too late.

In [ ]:
SYSTEM_PROMPT = "You answer questions about Acme inventory. Use the tools. Never guess numbers."


def run_agent(model, tools, task, max_steps=6, max_tokens=12000, max_repeats=2):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": task}]
    steps = 0
    tokens = 0
    times_seen = {}

    while True:
        if steps >= max_steps:
            return {"status": "max_steps", "steps": steps, "tokens": tokens, "messages": messages}
        if tokens >= max_tokens:
            return {"status": "token_budget", "steps": steps, "tokens": tokens, "messages": messages}

        reply = model(messages, tool_specs(tools))
        steps += 1
        tokens += reply.pop("tokens", 0)
        messages.append(reply)

        if not reply.get("tool_calls"):
            return {"status": "done", "answer": reply["content"],
                    "steps": steps, "tokens": tokens, "messages": messages}

        for tool_call in reply["tool_calls"]:
            signature = tool_call["function"]["name"] + tool_call["function"]["arguments"]
            times_seen[signature] = times_seen.get(signature, 0) + 1
            if times_seen[signature] > max_repeats:
                return {"status": "loop_detected", "steps": steps, "tokens": tokens, "messages": messages}

            result = run_tool(tool_call, tools)
            messages.append({"role": "tool", "tool_call_id": tool_call["id"], "content": result})

## 4. Test it with a fake model
No network and no tokens. Each check is one way an agent can go wrong.

In [ ]:
class FakeModel:
    """Plays back replies we prepared. Lets us test without network or tokens."""

    def __init__(self, replies):
        self.replies = list(replies)

    def __call__(self, messages, tools=None):
        return dict(self.replies.pop(0))


def tool_reply(name, **arguments):
    """A prepared reply in which the model asks for one tool call."""
    call = {"id": "call_" + uuid.uuid4().hex[:6], "type": "function",
            "function": {"name": name, "arguments": json.dumps(arguments)}}
    return {"role": "assistant", "content": None, "tool_calls": [call], "tokens": 300}


def text_reply(text):
    return {"role": "assistant", "content": text, "tokens": 300}

In [ ]:
model = FakeModel([tool_reply("get_product", sku="SKU-007"), text_reply("We have 79 units.")])
result = run_agent(model, TOOLS, "How many SKU-007?")
print("normal run:     ", result["status"], "| tool said:", result["messages"][3]["content"][:60])

model = FakeModel([tool_reply("delete_everything"), text_reply("ok")])
result = run_agent(model, TOOLS, "anything")
print("unknown tool:   ", result["messages"][3]["content"])

broken = tool_reply("get_product")
broken["tool_calls"][0]["function"]["arguments"] = '{"sku": "SKU-0'
result = run_agent(FakeModel([broken, text_reply("ok")]), TOOLS, "anything")
print("broken JSON:    ", result["messages"][3]["content"])

many_calls = [tool_reply("get_product", sku=f"SKU-00{i}") for i in range(1, 10)]
print("step budget:    ", run_agent(FakeModel(many_calls), TOOLS, "x", max_steps=3)["status"])
print("token budget:   ", run_agent(FakeModel(many_calls), TOOLS, "x", max_tokens=500)["status"])

same_call = [tool_reply("get_product", sku="SKU-001") for _ in range(9)]
print("repeated call:  ", run_agent(FakeModel(same_call), TOOLS, "x")["status"])

## 5. Now with the real model
*Before you run it:* how many model calls will this take? Which tool, with which argument?

In [ ]:
result = run_agent(ask_model, TOOLS, "Which products have fewer than 10 units in stock, and what does each cost?")
print(result["status"], "-", result["steps"], "model calls -", result["tokens"], "tokens")
print()
for message in result["messages"]:
    for tool_call in message.get("tool_calls") or []:
        print("tool:", tool_call["function"]["name"], tool_call["function"]["arguments"])
print()
print(result.get("answer"))

## 6. The loop is today's default, not the definition of an agent
Plan-then-execute, code-as-action and explicit graphs are also valid designs. Whatever you choose must still give you:

| Property | Where it is in our loop |
|---|---|
| **Bounded**: it always stops | step and token budgets, repeated-call check |
| **Observable**: every decision can be inspected | `messages` |
| **Interruptible**: it can pause and resume | not yet; Lab 4 needs it for approvals |
| **Attributable**: every action traces back to a step | `tool_call_id` |
| **Costed**: spend is limited and measured | `tokens` |

Discussion: which of these does your framework give you without extra work?

## Take away
- The model proposes, our code decides. A tool error is a message, not an exception.
- Check budgets before paying for the next call.
- Judge any agent design by these five properties, not by its name.